# Semantic Chunking with AWS

## 📖 Overview

This notebook demonstrates **Semantic Chunking** for RAG using AWS services:
- **Amazon Bedrock Titan** for generating embeddings
- **AWS OpenSearch Serverless** for vector storage
- **Amazon Bedrock Claude** for answer generation

### What is Semantic Chunking?

Semantic Chunking creates document chunks based on **meaning boundaries** rather than arbitrary character counts.

**Traditional Fixed-Size Chunking:**
```
Chunk 1: "AWS Bedrock provides access to foundation model..."
Chunk 2: "...els through a single API. Claude is Anthro..."
Chunk 3: "...opic's most capable model for complex task..."
```
❌ **Problems**: Splits mid-sentence, breaks context, unnatural boundaries

**Semantic Chunking:**
```
Chunk 1: "AWS Bedrock provides access to foundation models through a single API."
Chunk 2: "Claude is Anthropic's most capable model for complex tasks."
Chunk 3: "Vector embeddings capture semantic meaning for similarity search."
```
✅ **Benefits**: Preserves meaning, natural boundaries, better context

### How It Works

1. **Split**: Break text into small units (sentences)
2. **Embed**: Generate embeddings for each sentence
3. **Compare**: Calculate similarity between consecutive sentences
4. **Boundary**: Create chunk break when similarity drops significantly
5. **Group**: Combine sentences between boundaries into chunks

### When to Use

✅ **Good for:**
- Long, structured documents
- Technical content with distinct topics
- When preserving context critical
- Multi-topic documents
- Improving retrieval precision

❌ **Not ideal for:**
- Very short documents
- Unstructured stream-of-consciousness
- When speed is critical (slower than fixed)
- Simple keyword matching
- Tight budgets (more embeddings)

### Architecture

```mermaid
graph TB
    A[Long Document] --> B[Split into Sentences]
    B --> C[Generate Embeddings<br/>for Each Sentence]
    C --> D[Calculate Similarity<br/>Between Consecutive Sentences]
    D --> E[Find Semantic Boundaries<br/>Low Similarity Points]
    E --> F[Create Chunks<br/>Group by Boundaries]
    F --> G[Index Semantic Chunks<br/>OpenSearch]
    
    H[Query] --> I[Vector Search]
    G --> I
    I --> J[Retrieve Semantically<br/>Coherent Chunks]
    J --> K[Generate Answer<br/>Claude]

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#f3e5f5
    style D fill:#e8f5e9
    style E fill:#fff9c4
    style F fill:#ffe0b2
    style G fill:#c8e6c9
    style H fill:#e1f5ff
    style I fill:#f3e5f5
    style J fill:#e8f5e9
    style K fill:#c8e6c9
```

## 1️⃣ Setup & Imports

In [ ]:
import sys
import json
from typing import List, Dict, Tuple
import time
import numpy as np
import re

sys.path.append('..')

from aws_utils.opensearch_manager import OpenSearchManager
from aws_utils.bedrock_client import BedrockEmbeddings, BedrockLLM, BedrockRAG
from aws_utils.rag_evaluator import RAGEvaluator

print("✓ Imports successful")

## 2️⃣ Configuration

In [ ]:
# AWS Configuration
AWS_REGION = 'us-west-2'
FIXED_INDEX = 'fixed_chunks'
SEMANTIC_INDEX = 'semantic_chunks'

# Model Configuration
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'
LLM_MODEL = 'us.anthropic.claude-opus-4-1-20250805-v1:0'

# Chunking Parameters
FIXED_CHUNK_SIZE = 300  # Characters
FIXED_CHUNK_OVERLAP = 50

# Semantic Chunking Parameters
SIMILARITY_THRESHOLD = 0.75  # Cosine similarity threshold for boundaries
MIN_CHUNK_SIZE = 100  # Minimum characters per chunk
MAX_CHUNK_SIZE = 500  # Maximum characters per chunk

print(f"Configuration:")
print(f"  Fixed Chunking: {FIXED_CHUNK_SIZE} chars, {FIXED_CHUNK_OVERLAP} overlap")
print(f"  Semantic Chunking: Threshold {SIMILARITY_THRESHOLD}")
print(f"  Chunk size range: {MIN_CHUNK_SIZE}-{MAX_CHUNK_SIZE} chars")

## 3️⃣ Initialize Services

In [ ]:
fixed_opensearch = OpenSearchManager(region=AWS_REGION, index_name=FIXED_INDEX)
semantic_opensearch = OpenSearchManager(region=AWS_REGION, index_name=SEMANTIC_INDEX)
embedder = BedrockEmbeddings(AWS_REGION, EMBEDDING_MODEL)
llm = BedrockLLM(AWS_REGION, LLM_MODEL)

print("✓ Services initialized")

## 4️⃣ Load Sample Document

We'll use a long document with multiple distinct topics.

In [ ]:
sample_document = """
Amazon Web Services provides a comprehensive suite of cloud computing services for businesses worldwide. 
The platform includes compute, storage, database, analytics, machine learning, and many other services. 
AWS operates in multiple regions globally, providing low-latency access to services.

AWS Bedrock is a fully managed service that provides access to foundation models from leading AI companies. 
You can choose from models by Anthropic, Amazon, Cohere, Meta, and Stability AI through a single API. 
Bedrock enables you to build and scale generative AI applications with security and privacy features.

The Claude model family from Anthropic includes three variants with different capabilities. 
Claude Opus excels at complex analysis, mathematics, and multi-step reasoning tasks. 
Claude Sonnet provides balanced performance for enterprise workloads. 
Claude Haiku is optimized for speed and cost-effectiveness.

Retrieval-Augmented Generation (RAG) combines information retrieval with text generation. 
This approach grounds LLM responses in external knowledge sources, reducing hallucinations. 
RAG systems typically have three stages: indexing, retrieval, and generation.

Vector embeddings are numerical representations that capture semantic meaning. 
Amazon Titan Embeddings V2 produces 1024-dimensional vectors for semantic search. 
These vectors enable finding similar content by comparing positions in high-dimensional space. 
Cosine similarity is commonly used to measure the distance between embedding vectors.

OpenSearch Serverless automatically scales compute and storage resources. 
It eliminates the need to configure and manage OpenSearch clusters. 
The service supports vector search using algorithms like HNSW for efficient similarity search. 
Integration with AWS services like Lambda and Kinesis enables real-time data processing.

Optimizing RAG systems involves balancing multiple factors including quality, latency, and cost. 
Techniques include chunking strategies, hybrid search, reranking, and query optimization. 
Monitoring metrics like precision, recall, and answer faithfulness helps track performance. 
Caching frequently accessed embeddings and results can significantly reduce costs.

Production RAG deployments require careful consideration of infrastructure and operations. 
Implement proper error handling, rate limiting, and fallback mechanisms for reliability. 
Security measures should include encryption, IAM policies, and content filtering. 
Regular evaluation and A/B testing ensure the system continues meeting requirements.
"""

print(f"Document length: {len(sample_document)} characters")
print(f"Document length: {len(sample_document.split())} words")
print(f"\nDocument preview:\n{sample_document[:300]}...")

## 5️⃣ Fixed-Size Chunking (Baseline)

Traditional approach with fixed character counts.

In [ ]:
def fixed_size_chunking(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    """
    Split text into fixed-size chunks with overlap
    """
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        
        if chunk:
            chunks.append(chunk)
        
        start = end - overlap
    
    return chunks

# Create fixed chunks
fixed_chunks = fixed_size_chunking(
    sample_document,
    chunk_size=FIXED_CHUNK_SIZE,
    overlap=FIXED_CHUNK_OVERLAP
)

print(f"Fixed-Size Chunking Results:\n")
print(f"Number of chunks: {len(fixed_chunks)}")
print(f"Average chunk size: {sum(len(c) for c in fixed_chunks) / len(fixed_chunks):.0f} chars\n")

print("Sample chunks:\n")
for i, chunk in enumerate(fixed_chunks[:5], 1):
    print(f"Chunk {i} ({len(chunk)} chars):")
    print(f"{chunk[:100]}...\n")

## 6️⃣ Semantic Chunking Implementation

Create chunks based on semantic similarity.

### Algorithm Steps

1. **Split into sentences** using regex
2. **Generate embeddings** for each sentence
3. **Calculate similarities** between consecutive sentences
4. **Identify boundaries** where similarity < threshold
5. **Group sentences** into semantically coherent chunks

In [ ]:
def split_into_sentences(text: str) -> List[str]:
    """
    Split text into sentences using regex
    """
    # Simple sentence splitter (can be improved with spaCy/nltk)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

def cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
    """
    Calculate cosine similarity between two vectors
    """
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def semantic_chunking(text: str,
                     embedder: BedrockEmbeddings,
                     similarity_threshold: float = 0.75,
                     min_chunk_size: int = 100,
                     max_chunk_size: int = 500) -> List[Dict]:
    """
    Create chunks based on semantic similarity between sentences
    
    Returns:
        List of dicts with 'text' and 'metadata'
    """
    print("Step 1: Splitting into sentences...")
    sentences = split_into_sentences(text)
    print(f"✓ Found {len(sentences)} sentences\n")
    
    print("Step 2: Generating embeddings for each sentence...")
    embeddings = []
    for i, sentence in enumerate(sentences):
        emb = embedder.embed_text(sentence)
        embeddings.append(emb)
        if (i + 1) % 5 == 0:
            print(f"  Embedded {i + 1}/{len(sentences)} sentences")
    print(f"✓ Generated {len(embeddings)} embeddings\n")
    
    print("Step 3: Calculating similarities...")
    similarities = []
    for i in range(len(embeddings) - 1):
        sim = cosine_similarity(embeddings[i], embeddings[i + 1])
        similarities.append(sim)
    print(f"✓ Calculated {len(similarities)} similarities\n")
    
    print(f"Step 4: Finding boundaries (threshold: {similarity_threshold})...")
    boundaries = [0]  # Start with first sentence
    for i, sim in enumerate(similarities):
        if sim < similarity_threshold:
            boundaries.append(i + 1)
    boundaries.append(len(sentences))  # Add end
    print(f"✓ Found {len(boundaries) - 1} semantic boundaries\n")
    
    print("Step 5: Creating chunks...")
    chunks = []
    for i in range(len(boundaries) - 1):
        start_idx = boundaries[i]
        end_idx = boundaries[i + 1]
        
        chunk_sentences = sentences[start_idx:end_idx]
        chunk_text = ' '.join(chunk_sentences)
        
        # Enforce size constraints
        if len(chunk_text) < min_chunk_size:
            # Merge with next chunk if possible
            continue
        
        if len(chunk_text) > max_chunk_size:
            # Split large chunks
            # Simple split - could be more sophisticated
            mid = len(chunk_sentences) // 2
            chunk1 = ' '.join(chunk_sentences[:mid])
            chunk2 = ' '.join(chunk_sentences[mid:])
            
            if len(chunk1) >= min_chunk_size:
                chunks.append({
                    'text': chunk1,
                    'metadata': {
                        'chunk_id': len(chunks),
                        'sentence_count': mid,
                        'char_count': len(chunk1)
                    }
                })
            if len(chunk2) >= min_chunk_size:
                chunks.append({
                    'text': chunk2,
                    'metadata': {
                        'chunk_id': len(chunks),
                        'sentence_count': len(chunk_sentences) - mid,
                        'char_count': len(chunk2)
                    }
                })
        else:
            chunks.append({
                'text': chunk_text,
                'metadata': {
                    'chunk_id': len(chunks),
                    'sentence_count': len(chunk_sentences),
                    'char_count': len(chunk_text)
                }
            })
    
    print(f"✓ Created {len(chunks)} semantic chunks\n")
    return chunks, similarities

# Create semantic chunks
print("Semantic Chunking Process:\n" + "="*70 + "\n")
semantic_chunks, similarities = semantic_chunking(
    sample_document,
    embedder,
    similarity_threshold=SIMILARITY_THRESHOLD,
    min_chunk_size=MIN_CHUNK_SIZE,
    max_chunk_size=MAX_CHUNK_SIZE
)

print("\nSemantic Chunking Results:")
print(f"Number of chunks: {len(semantic_chunks)}")
print(f"Average chunk size: {sum(c['metadata']['char_count'] for c in semantic_chunks) / len(semantic_chunks):.0f} chars")
print(f"Average sentences per chunk: {sum(c['metadata']['sentence_count'] for c in semantic_chunks) / len(semantic_chunks):.1f}\n")

print("Sample chunks:\n")
for i, chunk in enumerate(semantic_chunks[:5], 1):
    meta = chunk['metadata']
    print(f"Chunk {i} ({meta['char_count']} chars, {meta['sentence_count']} sentences):")
    print(f"{chunk['text'][:100]}...\n")

## 7️⃣ Visualize Similarity Distribution

See where semantic boundaries occur.

In [ ]:
import matplotlib.pyplot as plt

# Plot similarities
plt.figure(figsize=(14, 6))

x = list(range(len(similarities)))
plt.plot(x, similarities, 'o-', linewidth=2, markersize=8, label='Sentence Similarity')
plt.axhline(y=SIMILARITY_THRESHOLD, color='red', linestyle='--', 
           linewidth=2, label=f'Threshold ({SIMILARITY_THRESHOLD})')

# Mark boundaries
boundaries_idx = [i for i, sim in enumerate(similarities) if sim < SIMILARITY_THRESHOLD]
if boundaries_idx:
    plt.scatter(boundaries_idx, [similarities[i] for i in boundaries_idx],
               color='red', s=200, marker='X', zorder=5, label='Semantic Boundaries')

plt.xlabel('Sentence Pair Index', fontsize=12, fontweight='bold')
plt.ylabel('Cosine Similarity', fontsize=12, fontweight='bold')
plt.title('Semantic Similarity Between Consecutive Sentences', 
         fontsize=14, fontweight='bold')
plt.legend(fontsize=11, loc='best')
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.0)
plt.tight_layout()
plt.show()

print(f"\n📊 Similarity Statistics:")
print(f"  Mean similarity: {np.mean(similarities):.3f}")
print(f"  Std deviation: {np.std(similarities):.3f}")
print(f"  Min similarity: {min(similarities):.3f}")
print(f"  Max similarity: {max(similarities):.3f}")
print(f"  Boundaries found: {len(boundaries_idx)}")

## 8️⃣ Index Both Chunking Strategies

In [ ]:
# Index fixed chunks
print("Indexing fixed-size chunks...")
fixed_opensearch.create_index(
    embedding_dim=embedder.get_embedding_dimension(),
    force_recreate=True
)

fixed_docs = []
for i, chunk in enumerate(fixed_chunks):
    embedding = embedder.embed_text(chunk)
    fixed_docs.append({
        'text': chunk,
        'embedding': embedding,
        'metadata': {'chunk_id': i, 'strategy': 'fixed'}
    })

fixed_opensearch.index_documents(fixed_docs)
print(f"✓ Indexed {len(fixed_docs)} fixed chunks\n")

# Index semantic chunks
print("Indexing semantic chunks...")
semantic_opensearch.create_index(
    embedding_dim=embedder.get_embedding_dimension(),
    force_recreate=True
)

semantic_docs = []
for chunk in semantic_chunks:
    embedding = embedder.embed_text(chunk['text'])
    semantic_docs.append({
        'text': chunk['text'],
        'embedding': embedding,
        'metadata': {**chunk['metadata'], 'strategy': 'semantic'}
    })

semantic_opensearch.index_documents(semantic_docs)
print(f"✓ Indexed {len(semantic_docs)} semantic chunks")

## 9️⃣ Compare Retrieval Quality

Test both strategies with sample queries.

In [ ]:
test_queries = [
    "What is AWS Bedrock?",
    "Explain Claude model variants",
    "How does RAG work?",
    "What are vector embeddings?"
]

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*70}")
    print(f"Query {i}: {query}")
    print('='*70)
    
    query_emb = embedder.embed_text(query)
    
    # Fixed chunking retrieval
    print(f"\n📦 Fixed-Size Chunking:")
    fixed_results = fixed_opensearch.vector_search(query_emb, top_k=3)
    for j, result in enumerate(fixed_results, 1):
        print(f"  {j}. [Score: {result['score']:.4f}] {result['text'][:80]}...")
    
    # Semantic chunking retrieval
    print(f"\n🧠 Semantic Chunking:")
    semantic_results = semantic_opensearch.vector_search(query_emb, top_k=3)
    for j, result in enumerate(semantic_results, 1):
        meta = result['metadata']
        print(f"  {j}. [Score: {result['score']:.4f}, {meta['sentence_count']} sents]")
        print(f"     {result['text'][:80]}...")
    
    # Generate answers
    fixed_answer = llm.generate_with_context(query, [r['text'] for r in fixed_results])
    semantic_answer = llm.generate_with_context(query, [r['text'] for r in semantic_results])
    
    print(f"\n📝 Answers:")
    print(f"\n  Fixed: {fixed_answer[:150]}...")
    print(f"\n  Semantic: {semantic_answer[:150]}...")

## 🔟 Chunk Quality Analysis

Compare chunk coherence and boundaries.

In [ ]:
print("Chunk Quality Comparison\n")
print("="*70)

# Fixed chunking stats
fixed_lengths = [len(c) for c in fixed_chunks]
print(f"\n📦 Fixed-Size Chunking:")
print(f"  Total chunks: {len(fixed_chunks)}")
print(f"  Avg length: {np.mean(fixed_lengths):.0f} chars")
print(f"  Std dev: {np.std(fixed_lengths):.0f} chars")
print(f"  Min/Max: {min(fixed_lengths)}/{max(fixed_lengths)} chars")

# Check for mid-sentence splits
mid_sentence_splits = sum(1 for c in fixed_chunks if not c[0].isupper())
print(f"  Mid-sentence splits: {mid_sentence_splits} ({mid_sentence_splits/len(fixed_chunks)*100:.1f}%)")

# Semantic chunking stats  
semantic_lengths = [c['metadata']['char_count'] for c in semantic_chunks]
print(f"\n🧠 Semantic Chunking:")
print(f"  Total chunks: {len(semantic_chunks)}")
print(f"  Avg length: {np.mean(semantic_lengths):.0f} chars")
print(f"  Std dev: {np.std(semantic_lengths):.0f} chars")
print(f"  Min/Max: {min(semantic_lengths)}/{max(semantic_lengths)} chars")

# All semantic chunks start with capital (sentence boundaries)
print(f"  Mid-sentence splits: 0 (0.0%)")
print(f"  Natural boundaries: {len(semantic_chunks)} (100.0%)")

# Visualize chunk size distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Fixed chunks
ax1.hist(fixed_lengths, bins=15, color='skyblue', edgecolor='navy', linewidth=1.5)
ax1.axvline(FIXED_CHUNK_SIZE, color='red', linestyle='--', linewidth=2, label='Target Size')
ax1.set_xlabel('Chunk Length (characters)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax1.set_title('Fixed-Size Chunking', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Semantic chunks
ax2.hist(semantic_lengths, bins=15, color='lightgreen', edgecolor='darkgreen', linewidth=1.5)
ax2.axvline(np.mean(semantic_lengths), color='red', linestyle='--', linewidth=2, label='Mean Size')
ax2.set_xlabel('Chunk Length (characters)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax2.set_title('Semantic Chunking', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 Analysis:")
print(f"  - Fixed chunking: Uniform size, but {mid_sentence_splits} broken sentences")
print(f"  - Semantic chunking: Variable size, but 100% natural boundaries")
print(f"  - Semantic preserves context within each chunk")
print(f"  - Fixed may split important information across chunks")

## 1️⃣1️⃣ Summary & Key Takeaways

### What We Built

✅ Fixed-size chunking (baseline)
✅ Semantic chunking algorithm
✅ Sentence embedding and similarity calculation
✅ Boundary detection based on similarity
✅ Comparison of both strategies
✅ Quality analysis and visualization

### Performance Characteristics

**Fixed-Size Chunking:**
- **Speed**: Very fast (no embeddings needed)
- **Cost**: Minimal
- **Quality**: May split context
- **Consistency**: Uniform sizes

**Semantic Chunking:**
- **Speed**: Slower (requires embeddings)
- **Cost**: Higher (N sentence embeddings)
- **Quality**: Preserves context
- **Consistency**: Variable sizes

### When to Use Semantic Chunking

**Use Semantic when:**
- Documents have clear topic transitions
- Context preservation is critical
- Building high-quality knowledge bases
- Documents are long and structured
- Retrieval precision matters most

**Use Fixed when:**
- Speed and simplicity needed
- Budget is tight
- Documents are short
- Uniform chunk sizes required
- Prototyping/development

### Key Insights

1. **Natural Boundaries**: Semantic chunking respects meaning boundaries
2. **Context Preservation**: Entire topics stay together
3. **Better Retrieval**: Chunks are self-contained and coherent
4. **Trade-off**: Quality vs speed/cost
5. **Threshold Matters**: Lower threshold = more, smaller chunks

### Optimization Strategies

**1. Caching Embeddings**
```python
# Cache sentence embeddings for reuse
sentence_cache = {}
for sent in sentences:
    if sent not in sentence_cache:
        sentence_cache[sent] = embedder.embed_text(sent)
```

**2. Batch Processing**
```python
# Generate embeddings in batches
embeddings = embedder.embed_batch(sentences, batch_size=10)
```

**3. Adaptive Thresholds**
```python
# Use percentile-based threshold
threshold = np.percentile(similarities, 25)  # Bottom 25%
```

**4. Hybrid Approach**
```python
# Use semantic for long docs, fixed for short
if len(doc) > 1000:
    chunks = semantic_chunking(doc)
else:
    chunks = fixed_chunking(doc)
```

### Advanced Techniques

**1. Multi-level Chunking**
- Paragraph-level semantic boundaries
- Sentence-level within paragraphs
- Hierarchical structure

**2. Topic Modeling**
- Use LDA or NMF for topics
- Create chunks per topic
- Better for multi-topic documents

**3. Sliding Window**
- Compare sentence to next N sentences
- More robust to local variations
- Smoother boundary detection

**4. Learned Boundaries**
- Train model to predict boundaries
- Fine-tune on domain data
- Best quality but complex

### Best Practices

1. **Set Size Constraints**: Min/max chunk sizes prevent extremes
2. **Test Thresholds**: Tune similarity threshold for your domain
3. **Visualize**: Plot similarities to understand boundaries
4. **Evaluate**: Measure retrieval quality, not just chunk count
5. **Consider Cost**: Balance quality with embedding costs

### Limitations

1. **Slower**: Requires embedding every sentence
2. **More Expensive**: N sentence embeddings vs M chunk embeddings
3. **Threshold Tuning**: Requires experimentation
4. **Variable Sizes**: May not fit all systems
5. **Not Universal**: Works best for structured text

### Combining with Other Patterns

**Semantic Chunking + Reranking:**
- Better initial chunks
- Rerank for precision
- Best quality

**Semantic Chunking + Compression:**
- Natural chunk boundaries
- Compress within chunks
- Removes redundancy

**Semantic Chunking + Hierarchical:**
- Parent chunks (topics)
- Child chunks (sentences)
- Multi-level retrieval

### Alternative Sentence Splitters

Our regex approach is simple. For production:

```python
# spaCy (better)
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
sentences = [sent.text for sent in doc.sents]

# NLTK (alternative)
import nltk
sentences = nltk.sent_tokenize(text)
```

### Next Steps

- **Try different thresholds**: 0.5, 0.7, 0.8, 0.9
- **Add overlap**: Include last sentence of previous chunk
- **Hierarchical chunking**: Topics → paragraphs → sentences
- **Cross-document**: Find similar chunks across documents
- **Dynamic adjustment**: Learn optimal threshold per document type

## 🧹 Cleanup

In [ ]:
# Uncomment to delete indices
# fixed_opensearch.delete_index(FIXED_INDEX)
# semantic_opensearch.delete_index(SEMANTIC_INDEX)
# print("✓ Deleted both indices")

print("\nTo delete indices later:")
print(f"  fixed_opensearch.delete_index('{FIXED_INDEX}')")
print(f"  semantic_opensearch.delete_index('{SEMANTIC_INDEX}')")